# 04 — Filter-Bank CCA (FBCCA)

**Owner:** shared
**Reference:** Chen, X., Wang, Y., Gao, S., Jung, T.-P., & Gao, X. (2015). *Filter bank canonical correlation analysis for implementing a high-speed SSVEP-based brain-computer interface.* J. Neural Eng. 12:046008.

**This notebook is a pedagogical wrapper** around `src/ssvep/classifiers/fbcca.py`. It does not reimplement FBCCA; it walks through the sub-band design (§3.2.1 M3 of Chen 2015), the per-sub-band CCA, the squared-correlation aggregation with the `n^-1.25 + 0.25` weights, and finally runs `FBCCAClassifier` under LOBO.

FBCCA's core idea: SSVEP responses contain harmonic energy, but standard CCA's single ρ per class can't weight the fundamental and harmonics differently. FBCCA decomposes the EEG into a bank of sub-bands (each isolating a different harmonic range), runs CCA per sub-band, and combines per-class scores with weights that decay with sub-band index. The weighting empirically matches SSVEP spectral characteristics.

**Anchor (must reproduce):** mean = 0.950 ± 0.061 at 3.0 s window, H=2 — `results/tables/comparison_3s.csv` FBCCA row.


In [1]:
import os
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'raw').exists()), None)
if _root and Path.cwd() != _root:
    os.chdir(_root)

import numpy as np
import scipy.signal as sps
import matplotlib.pyplot as plt

from ssvep.io import load_all
from ssvep.classifiers import FBCCAClassifier
from ssvep.evaluation import leave_one_block_out_cv, itr


## 1. Load data

Same canonical preprocessing as notebook 03.


In [2]:
d = load_all(window_s=3.0)
X, y, blocks = d['X'], d['y'], d['blocks']
fs, sf = d['fs'], d['stim_freqs']
print(f"X.shape: {X.shape}; fs={fs}; stim_freqs={sf}")


X.shape: (80, 8, 768); fs=256.0; stim_freqs=[ 9. 10. 12. 15.]


## 2. Filter bank design (Chen 2015 §3.2.1 M3)

Five sub-bands, each a Chebyshev I bandpass:

- Sub-band `n` (1-indexed): passband `[6 + 8(n−1), 90]` Hz, stopband `[4 + 8(n−1), 100]` Hz
- gpass = 3 dB, gstop = 40 dB, ripple Rp = 0.5 dB
- Applied via `scipy.signal.filtfilt` (zero-phase)

Each successive sub-band starts higher in the spectrum, so sub-band 1 captures the fundamentals (passband 6–90 Hz includes 9, 10, 12, 15 Hz), sub-band 2 captures the 2nd-harmonic range (passband 14–90 Hz), and so on.


In [3]:
def build_subband_filters(fs, num_subbands=5):
    nyq = fs / 2.0
    flt = []
    for n in range(1, num_subbands + 1):
        wp = ((6 + 8*(n-1))/nyq, 90.0/nyq)
        ws = ((4 + 8*(n-1))/nyq, 100.0/nyq)
        N, Wn = sps.cheb1ord(wp, ws, gpass=3, gstop=40)
        b, a = sps.cheby1(N, rp=0.5, Wn=Wn, btype='bandpass')
        flt.append((b, a, N))
    return flt

filters = build_subband_filters(fs, num_subbands=5)
print(f"{'Sub-band':<10}{'Passband (Hz)':<22}{'Order':<6}")
for n, (_, _, N) in enumerate(filters, start=1):
    print(f"  {n:<8}[{6+8*(n-1):>2}, 90]{'':<12}{N}")

# Plot frequency response of each sub-band
fig, ax = plt.subplots(figsize=(9, 3.5))
w = np.linspace(0, 0.5, 1024) * 2 * np.pi
for n, (b, a, _) in enumerate(filters, start=1):
    w_, h = sps.freqz(b, a, worN=w)
    ax.plot((w_ / (2*np.pi)) * fs, 20 * np.log10(np.abs(h) + 1e-12),
            label=f'sub-band {n} ([{6+8*(n-1)}, 90] Hz)')
ax.set_xlim(0, 100); ax.set_ylim(-80, 5)
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Magnitude (dB)')
ax.axhline(-3, color='gray', ls=':', lw=0.7, alpha=0.6)
ax.legend(fontsize=8); ax.set_title('FBCCA sub-band filter responses (Chen 2015 M3)')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


Sub-band  Passband (Hz)         Order 
  1       [ 6, 90]            6
  2       [14, 90]            9
  3       [22, 90]            11
  4       [30, 90]            12
  5       [38, 90]            12


C:\Users\cisha\AppData\Local\Temp\ipykernel_175564\1529658181.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3. Sub-band weights

Chen 2015 Equation 7 specifies `w(n) = n^-1.25 + 0.25` and aggregates per-class scores as `ρ_class = Σ_n w(n) · ρ_{n,class}²` (squared!). The weighting strongly favors the fundamental and 2nd harmonic.


In [4]:
weights = np.array([n ** -1.25 + 0.25 for n in range(1, 6)])
print(f"Sub-band weights w(n) = n^-1.25 + 0.25:")
for n, w_n in enumerate(weights, start=1):
    print(f"  w({n}) = {w_n:.4f}")


Sub-band weights w(n) = n^-1.25 + 0.25:
  w(1) = 1.2500
  w(2) = 0.6704
  w(3) = 0.5033
  w(4) = 0.4268
  w(5) = 0.3837


## 4. Per-sub-band CCA on one trial

Pick one trial (class 1 = 10 Hz). For each sub-band: filter, then run CCA against each candidate-frequency reference. Show the 5 (sub-bands) × 4 (classes) matrix of squared canonical correlations and the weighted-sum aggregation that produces the final argmax.


In [5]:
from sklearn.cross_decomposition import CCA as SklearnCCA
from ssvep.features import cca_reference_signals

trial_idx = int(np.where((y == 1) & (blocks == 0))[0][0])
x = X[trial_idx]                       # (8, 768)
n_samples = x.shape[-1]
all_refs = cca_reference_signals(sf, n_harmonics=2, fs=fs, n_samples=n_samples)

rho_sq = np.zeros((5, len(sf)))
for b_idx, (bb, aa, _) in enumerate(filters):
    X_b = sps.filtfilt(bb, aa, x, axis=-1)
    for k in range(len(sf)):
        cca = SklearnCCA(n_components=1, max_iter=500)
        u, v = cca.fit_transform(X_b.T, all_refs[k].T)
        rho = float(np.corrcoef(u[:, 0], v[:, 0])[0, 1])
        rho_sq[b_idx, k] = rho ** 2

print(f"trial #{trial_idx} — true class = {int(y[trial_idx])} ({sf[int(y[trial_idx])]} Hz)\n")
print(f"{'sub-band':<10}{'9 Hz':>10}{'10 Hz':>10}{'12 Hz':>10}{'15 Hz':>10}")
for b_idx in range(5):
    print(f"  {b_idx+1:<8}", end='')
    for k in range(4):
        print(f"{rho_sq[b_idx, k]:>10.4f}", end='')
    print()

aggregated = (weights[:, None] * rho_sq).sum(axis=0)
print(f"\nweighted Σ:  ", end='')
for k in range(4):
    print(f"{aggregated[k]:>10.4f}", end='')
print()
print(f"\nargmax → predicted class {int(np.argmax(aggregated))} ({sf[int(np.argmax(aggregated))]} Hz)")


trial #2 — true class = 1 (10.0 Hz)

sub-band        9 Hz     10 Hz     12 Hz     15 Hz
  1           0.0920    0.5930    0.1039    0.0995
  2           0.0964    0.4063    0.0449    0.2482
  3           0.0015    0.0026    0.0884    0.3631
  4           0.0006    0.0007    0.0011    0.4188
  5           0.0001    0.0001    0.0002    0.0003

weighted Σ:      0.1807    1.0153    0.2050    0.6524

argmax → predicted class 1 (10.0 Hz)


## 5. Cross-subject 4-block LOBO

`FBCCAClassifier` (`fit` is a no-op) under the standard 4-block LOBO. The default `num_subbands=5` matches Chen 2015 M3; we pass `num_harmonics=2` to match the headline harness configuration in `scripts/compare_classifiers.py`.

**Anchor:** mean = 0.950 ± 0.061, per_block = `{0: 1.0, 1: 1.0, 2: 0.85, 3: 0.95}`.


In [6]:
factory = lambda: FBCCAClassifier(stim_freqs=sf, fs=fs, num_harmonics=2, num_subbands=5)
result = leave_one_block_out_cv(X, y, blocks, factory)
print(f"per-block accuracies: {result['per_block']}")
print(f"mean ± std         : {result['mean']:.3f} ± {result['std']:.3f}")
print(f"ITR @ 3.0s window  : {itr(result['mean'], n_classes=4, window_s=3.0):.2f} bits/min")


per-block accuracies: {0: 1.0, 1: 1.0, 2: 0.85, 3: 0.95}
mean ± std         : 0.950 ± 0.061
ITR @ 3.0s window  : 32.69 bits/min


## 6. Multi-window comparison

FBCCA wins at every window length. The accuracy floor at 1 s already passes the 0.80 target — the dominant ITR cost is the window length itself, not the classifier.

Anchors from `comparison_*s.csv` FBCCA rows: 0.800 (1 s), 0.938 (2 s), 0.950 (3 s), 0.975 (5 s).


In [7]:
rows = []
for win in [1.0, 2.0, 3.0, 5.0]:
    ds = load_all(window_s=win)
    fac = lambda: FBCCAClassifier(stim_freqs=ds['stim_freqs'], fs=ds['fs'], num_harmonics=2, num_subbands=5)
    r = leave_one_block_out_cv(ds['X'], ds['y'], ds['blocks'], fac)
    rows.append((win, r['mean'], r['std'], itr(r['mean'], 4, win)))

print(f"{'window (s)':<12}{'accuracy':<14}{'std':<10}{'ITR (bpm)':<12}")
for win, m, s, i in rows:
    print(f"  {win:<10g}{m:<14.3f}{s:<10.3f}{i:<12.2f}")


window (s)  accuracy      std       ITR (bpm)   
  1         0.800         0.154     57.66       
  2         0.938         0.065     46.91       
  3         0.950         0.061     32.69       
  5         0.975         0.025     21.50       


## 7. Why harmonic recovery closes the universality gap

Subject 1 already saturates CCA at the fundamental (notebook 03 hits 1.000), so harmonic content there contributes nothing — at H=2, FBCCA scores 0.975/1.000 on subject 1 (the 0.025 dip is filter-bank noise on already-clean data, well within tolerance).

Subject 2 is the interesting case. Subject 2's fundamental-frequency SNR is weaker, but the harmonic content is still present and proportionally more informative. FBCCA's weighted sum `Σ_n w(n) · ρ_{n,class}²` recovers that signal:

| | CCA (H=2) | FBCCA (H=2) | Δ |
|---|---|---|---|
| Subject 1 | 1.000 | 1.000 | +0.000 |
| Subject 2 | 0.675 | 0.900 | **+0.225** |

(within-subject 3 s LOBO, from `comparison_within_subject_3s.csv`)

The 3.7 pp mean lift over CCA is concentrated entirely on subject 2 — this is what "harmonic recovery closes the universality gap" looks like quantitatively. See `results/within_subject_evaluation.md` §"Mechanism — Why FBCCA Wins" for the full analysis.
